In [1]:
!pip install -q pyspark

In [ ]:
!pip install -q pyspark

from google.colab import files
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark import StorageLevel


spark = SparkSession.builder \
    .appName("StreamingSet4") \
    .getOrCreate()

uploaded = files.upload()
def read_csv(name):
    return spark.read.option("header", "true").option("inferSchema", "true").csv(name)

sessions_df = read_csv("watch_sessions_80000.csv")
extra_df = read_csv("watch_sessions_extra_10000.csv")
subscribers_df = read_csv("subscriber_master_2400.csv")
content_df = read_csv("content_catalog_120.csv")
devices_df = read_csv("device_master_8.csv")
plans_df = read_csv("subscription_plans_4.csv")

print("Main sessions count:", sessions_df.count())
sessions_df.printSchema()
sessions_df.show(5, truncate=False)

In [ ]:
sessions_df.printSchema()
print("Row count:", sessions_df.count())
sessions_df.show(10, truncate=False)

In [ ]:
task2_df = sessions_df.filter(
    (F.col("session_status") == "Completed") & (F.col("watch_minutes") > 60)
).select("session_id", "subscriber_id", "content_id", "watch_minutes", "session_status")

task2_df.show(10, truncate=False)

In [ ]:
task3_df = sessions_df.withColumn(
    "watch_hours", F.col("watch_minutes") / 60.0
).withColumn(
    "high_engagement",
    F.when(F.col("completion_pct").isNull(), None)
     .when(F.col("completion_pct") >= 75.0, True)
     .otherwise(False)
)

task3_df.select("session_id", "watch_minutes", "watch_hours", "completion_pct", "high_engagement").show(10, truncate=False)

In [ ]:
task4_df = sessions_df.orderBy(F.col("data_used_mb").desc())
task4_df.show(20, truncate=False)

In [ ]:
cols_to_check = ["subscriber_id", "device_id", "completion_pct", "buffering_seconds", "user_rating"]

missing_exprs = [F.sum(F.col(c).isNull().cast("int")).alias(f"{c}_missing") for c in cols_to_check]
sessions_df.select(missing_exprs).show(truncate=False)

In [ ]:
task6_df = sessions_df.na.fill({"buffering_seconds": 0.0})
task6_df.select("session_id", "buffering_seconds").show(10, truncate=False)

In [ ]:
original_count = sessions_df.count()
task7_df = sessions_df.filter(F.col("device_id").isNotNull())
filtered_count = task7_df.count()

print(f"Original count: {original_count}")
print(f"Filtered count (non-null device_id): {filtered_count}")
print(f"Dropped rows: {original_count - filtered_count}")

In [ ]:
task8_df = sessions_df.withColumn(
    "completion_band",
    F.when(F.col("completion_pct").isNull(), "Unknown")
     .when(F.col("completion_pct") < 30, "Low")
     .when((F.col("completion_pct") >= 30) & (F.col("completion_pct") < 90), "Medium")
     .otherwise("High")
)

task8_df.select("session_id", "completion_pct", "completion_band").show(10, truncate=False)

In [ ]:
task9_df = sessions_df.join(content_df, "content_id") \
    .groupBy("genre") \
    .agg(
        F.count("session_id").alias("session_count"),
        F.sum("watch_minutes").alias("total_watch_minutes")
    ) \
    .orderBy(F.col("total_watch_minutes").desc())

task9_df.show(10, truncate=False)

In [ ]:
task10_df = sessions_df.join(devices_df, "device_id") \
    .groupBy("device_type") \
    .agg(
        F.avg("buffering_seconds").alias("avg_buffering_seconds"),
        F.avg("watch_minutes").alias("avg_watch_minutes")
    ) \
    .orderBy("device_type")

task10_df.show(truncate=False)

In [ ]:
task11_df = sessions_df.filter(F.col("subscriber_id").isNotNull()) \
    .join(subscribers_df, "subscriber_id") \
    .join(plans_df, "plan_id") \
    .groupBy("plan_name") \
    .agg(
        (F.sum(F.when(F.col("session_status") == "Completed", 1).otherwise(0)) / F.count("session_id")).alias("completion_rate"),
        F.count("session_id").alias("total_sessions")
    )

task11_df.show(truncate=False)

In [ ]:
task12_df = subscribers_df.filter(F.col("subscriber_status") == "Active") \
    .join(plans_df, "plan_id") \
    .groupBy("plan_id", "plan_name", "monthly_fee_inr") \
    .agg(
        F.count("subscriber_id").alias("active_subscriber_count")
    ) \
    .withColumn("mrr", F.col("active_subscriber_count") * F.col("monthly_fee_inr")) \
    .orderBy("plan_id")

task12_df.show(truncate=False)

In [ ]:
unmatched_subs = sessions_df.filter(F.col("subscriber_id").isNotNull()) \
    .join(subscribers_df, "subscriber_id", "left_anti").count()

unmatched_devices = sessions_df.filter(F.col("device_id").isNotNull()) \
    .join(devices_df, "device_id", "left_anti").count()

print(f"Unmatched subscriber_id count: {unmatched_subs}")
print(f"Unmatched device_id count: {unmatched_devices}")

In [ ]:
sessions_ts_df = sessions_df.withColumn("session_timestamp", F.to_timestamp("session_start")) \
                            .withColumn("year", F.year("session_timestamp")) \
                            .withColumn("month", F.month("session_timestamp"))

task14_df = sessions_ts_df.filter(F.col("year") == 2026) \
    .groupBy("year", "month") \
    .agg(
        F.count("session_id").alias("monthly_session_count"),
        F.sum("watch_minutes").alias("total_watch_minutes")
    ) \
    .orderBy("year", "month")

task14_df.show(truncate=False)

In [ ]:
task15_df = sessions_df.filter(F.col("subscriber_id").isNotNull()) \
    .join(subscribers_df, "subscriber_id") \
    .withColumn("session_ts", F.to_timestamp("session_start")) \
    .withColumn("signup_ts", F.to_timestamp("signup_date")) \
    .withColumn("days_since_signup", F.datediff(F.col("session_ts"), F.col("signup_ts"))) \
    .orderBy(F.col("session_ts").asc())

task15_df.select("session_id", "subscriber_id", "signup_date", "session_start", "days_since_signup").show(10, truncate=False)

In [ ]:
task16_df = sessions_df.withColumn("session_ts", F.to_timestamp("session_start")) \
    .withColumn("day_of_week", F.dayofweek("session_ts")) \
    .withColumn(
        "day_type",
        F.when(F.col("day_of_week").isin(1, 7), "Weekend").otherwise("Weekday")
    ) \
    .groupBy("day_type") \
    .agg(
        F.avg("watch_minutes").alias("avg_watch_minutes"),
        F.count("session_id").alias("session_count")
    )

task16_df.show(truncate=False)

In [ ]:
genre_content_agg = sessions_df.join(content_df, "content_id") \
    .groupBy("genre", "content_title") \
    .agg(F.sum("watch_minutes").alias("total_watch_minutes"))

window_genre = Window.partitionBy("genre").orderBy(F.col("total_watch_minutes").desc())

task17_df = genre_content_agg.withColumn("rank", F.dense_rank().over(window_genre)) \
    .filter(F.col("rank") <= 3) \
    .orderBy("genre", "rank")

task17_df.show(20, truncate=False)

In [ ]:
window_sub = Window.partitionBy("subscriber_id").orderBy(F.col("session_start").desc())

task18_df = sessions_df.filter(F.col("subscriber_id").isNotNull()) \
    .withColumn("rn", F.row_number().over(window_sub)) \
    .filter(F.col("rn") == 1) \
    .drop("rn")

task18_df.select("subscriber_id", "session_id", "session_start").show(10, truncate=False)

In [ ]:
window_lag = Window.partitionBy("subscriber_id").orderBy(F.col("session_start"), F.col("session_id"))

task19_df = sessions_df.filter(F.col("subscriber_id").isNotNull()) \
    .withColumn("prev_watch_minutes", F.lag("watch_minutes", 1).over(window_lag)) \
    .withColumn("diff_watch_minutes", F.col("watch_minutes") - F.col("prev_watch_minutes"))

task19_df.select("subscriber_id", "session_id", "session_start", "watch_minutes", "prev_watch_minutes", "diff_watch_minutes").show(10, truncate=False)

In [ ]:
combined_df = sessions_df.unionByName(extra_df)
total_count = combined_df.count()

dedup_df = combined_df.dropDuplicates(["session_id"])
dedup_count = dedup_df.count()

print(f"Total rows after union: {total_count}")
print(f"Total rows after deduplication: {dedup_count}")
print(f"Duplicates removed: {total_count - dedup_count}")

In [ ]:
pipeline_df = sessions_df.filter(F.col("session_status") == "Completed") \
    .join(content_df, "content_id") \
    .withColumn("watch_hours", F.col("watch_minutes") / 60.0) \
    .groupBy("genre") \
    .agg(F.sum("watch_hours").alias("total_watch_hours")) \
    .orderBy(F.col("total_watch_hours").desc())

pipeline_df.explain(True)
pipeline_df.show(truncate=False)

In [ ]:
import time

completed_sessions = sessions_df.filter(F.col("session_status") == "Completed")
completed_sessions.cache()
start = time.time()
print("Count 1:", completed_sessions.count())
print("Duration 1:", time.time() - start)
start = time.time()
print("Count 2:", completed_sessions.count())
print("Duration 2:", time.time() - start)

completed_sessions.unpersist()

In [ ]:
fourk_sessions = sessions_df.filter(F.col("stream_quality") == "4k")
fourk_sessions.persist(StorageLevel.MEMORY_AND_DISK)

print("4k count action 1:", fourk_sessions.count())
print("4k distinct devices action 2:", fourk_sessions.select("device_id").distinct().count())

fourk_sessions.unpersist()

In [ ]:
repartitioned_df = sessions_df.repartition(8)
partition_counts = repartitioned_df.groupBy(F.spark_partition_id().alias("partition_id")) \
    .agg(F.count("*").alias("row_count")) \
    .orderBy("partition_id")

partition_counts.show(truncate=False)

In [ ]:
print("--- Coalesce(2) Explain ---")
repartitioned_df.coalesce(2).explain(True)

print("--- Repartition(2) Explain ---")
repartitioned_df.repartition(2).explain(True)
repartitioned_df.coalesce(2).write.mode("overwrite").parquet("sessions_output.parquet")

***Capstone Project***

In [ ]:
all_sessions = sessions_df.unionByName(extra_df).dropDuplicates(["session_id"])
cleaned_sessions = all_sessions.filter(
    F.col("subscriber_id").isNotNull() &
    F.col("content_id").isNotNull() &
    F.col("device_id").isNotNull() &
    F.col("watch_minutes").isNotNull()
)
enriched_df = cleaned_sessions \
    .join(subscribers_df, "subscriber_id") \
    .join(content_df, "content_id") \
    .join(devices_df, "device_id") \
    .join(plans_df, "plan_id")
enriched_df = enriched_df.withColumn("session_ts", F.to_timestamp("session_start")) \
                         .withColumn("year", F.year("session_ts")) \
                         .withColumn("month", F.month("session_ts")) \
                         .withColumn("watch_hours", F.col("watch_minutes") / 60.0)

monthly_metrics = enriched_df.groupBy("region", "genre", "year", "month").agg(
    F.count("session_id").alias("session_count"),
    F.sum("watch_hours").alias("total_watch_hours"),
    (F.sum(F.when(F.col("session_status") == "Completed", 1).otherwise(0)) / F.count("session_id")).alias("completion_rate"),
    F.avg("buffering_seconds").alias("avg_buffering_seconds"),
    F.sum("data_used_mb").alias("total_data_used_mb")
)
genre_title_watch = enriched_df.groupBy("genre", "content_title").agg(
    F.sum("watch_hours").alias("content_watch_hours")
)

window_top_titles = Window.partitionBy("genre").orderBy(F.col("content_watch_hours").desc())
top_content_df = genre_title_watch.withColumn("rank", F.dense_rank().over(window_top_titles)) \
    .filter(F.col("rank") <= 3)
monthly_metrics.cache()
print("Capstone Action 1 (Count):", monthly_metrics.count())
print("Capstone Action 2 (Top 5 rows):")
monthly_metrics.show(5, truncate=False)
monthly_metrics.unpersist()
monthly_metrics.repartition(3).write.mode("overwrite").parquet("streaming_performance_report.parquet")
print("Streaming performance report successfully generated and saved to Parquet!")